# Download Results from Google Drive
Downloads all JSON files from Google Drive (TTS and Waveform folders) and recreates the local folder structure.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
from pathlib import Path

# Define source and destination paths
drive_base = '/content/drive/MyDrive/results'
local_base = '/content/results'

# Create local base directory
Path(local_base).mkdir(parents=True, exist_ok=True)

# Define source folders
source_folders = ['TTS', 'Waveform']

print(f"Source base path: {drive_base}")
print(f"Local base path: {local_base}")
print(f"\nStarting download...\n")

In [ ]:
# Process each folder (TTS and Waveform)
total_files = 0
total_size = 0

for folder_type in source_folders:
    source_path = os.path.join(drive_base, folder_type)
    dest_path = os.path.join(local_base, folder_type)
    
    print(f"\n{'='*60}")
    print(f"Processing: {folder_type}")
    print(f"Source: {source_path}")
    print(f"Destination: {dest_path}")
    print(f"{'='*60}")
    
    if not os.path.exists(source_path):
        print(f"⚠️  Source path does not exist: {source_path}")
        continue
    
    # Walk through the directory structure
    for root, dirs, files in os.walk(source_path):
        # Get the relative path from source
        rel_path = os.path.relpath(root, source_path)
        
        # Create corresponding destination directory
        if rel_path == '.':
            current_dest = dest_path
        else:
            current_dest = os.path.join(dest_path, rel_path)
        
        Path(current_dest).mkdir(parents=True, exist_ok=True)
        
        # Copy JSON files
        json_files = [f for f in files if f.endswith('.json')]
        
        for file in json_files:
            source_file = os.path.join(root, file)
            dest_file = os.path.join(current_dest, file)
            
            # Copy the file
            shutil.copy2(source_file, dest_file)
            
            # Get file size
            file_size = os.path.getsize(dest_file)
            total_files += 1
            total_size += file_size
            
            # Print relative path
            relative_file_path = os.path.relpath(dest_file, local_base)
            print(f"✓ {relative_file_path} ({file_size / 1024:.1f} KB)")

print(f"\n{'='*60}")
print(f"Download Complete!")
print(f"{'='*60}")
print(f"Total files downloaded: {total_files}")
print(f"Total size: {total_size / (1024*1024):.2f} MB")
print(f"Local path: {local_base}")

In [ ]:
# Verify the downloaded structure
print("\nVerifying downloaded structure...\n")

for root, dirs, files in os.walk(local_base):
    level = root.replace(local_base, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    
    subindent = ' ' * 2 * (level + 1)
    json_count = sum(1 for f in files if f.endswith('.json'))
    if json_count > 0:
        print(f"{subindent}({json_count} JSON files)")

In [ ]:
import shutil

# Create zip file
zip_filename = 'results'
zip_path = f'/content/{zip_filename}'

print(f"Creating zip file: {zip_filename}.zip\n")
shutil.make_archive(zip_path, 'zip', '/content', 'results')

zip_size = os.path.getsize(f'{zip_path}.zip') / (1024*1024)
print(f"\n{'='*60}")
print(f"✓ Zip file created!")
print(f"{'='*60}")
print(f"Filename: {zip_filename}.zip")
print(f"Size: {zip_size:.2f} MB")
print(f"Path: {zip_path}.zip")
print(f"\nThe file is ready to download from Colab's file panel (left sidebar).")